# Домашнее задание: Однофакторный дисперсионный анализ (ANOVA)

## Импорты


In [ ]:
# дополняйте ячейку любыми импортами по желанию!
import numpy as np
import pandas as pd
import scipy.stats as st
from scipy.stats import norm, t, f, chi2, shapiro, levene, bartlett
from scipy.stats import ttest_1samp, ttest_ind, ttest_rel
from scipy.stats import wilcoxon
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.stats.power import FTestAnovaPower
from statsmodels.stats.oneway import anova_oneway
from statsmodels.stats.multicomp import pairwise_tukeyhsd
try:
    import pingouin as pg
    PINGOUIN_AVAILABLE = True
except ImportError:
    PINGOUIN_AVAILABLE = False
    print("pingouin не установлен, используем альтернативные методы")

try:
    import scikit_posthocs as sp
    POSTHOCS_AVAILABLE = True
except ImportError:
    POSTHOCS_AVAILABLE = False
    print("scikit-posthocs не установлен, используем альтернативные методы")

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)


## Задание 1. Сопоставление статистик

Вам предложены описания ситуаций и статистических процедур. Для каждой ситуации укажите наилучшую применимую статистику из списка: z-test, t-test, Welch t-test, F-test (сравнение дисперсий), χ² (goodness-of-fit), ANOVA (one-way), Tukey HSD, Levene.

**Ситуации:**

1. Необходимо проверить, отличается ли средний объём дозы от заданного значения, известна дисперсия производства (укажите двусторонний/односторонний тест).

2. Сравнить средние двух независимых групп при неизвестных, но равных дисперсиях.

3. Сравнить средние двух независимых групп при неизвестных и неравных дисперсиях.

4. Оценить, одинаковы ли средние в трёх и более группах.

5. Проверить, согласуются ли наблюдаемые частоты с распределением Пуассона.

**Требуется: кратко объясните выбор для каждой ситуации (1—2 предложения).**


### Ответы на Задание 1:

**1. Необходимо проверить, отличается ли средний объём дозы от заданного значения, известна дисперсия производства.**

**Ответ: z-test (двусторонний или односторонний в зависимости от формулировки гипотезы)**

**Обоснование:** Когда известна генеральная дисперсия σ² и проверяется гипотеза о среднем значении, используется z-тест. Статистика z = (X̄ - μ₀) / (σ/√n) имеет стандартное нормальное распределение. Двусторонний тест используется, если проверяется простое отличие от заданного значения; односторонний — если проверяется направленное отклонение (больше или меньше).

---

**2. Сравнить средние двух независимых групп при неизвестных, но равных дисперсиях.**

**Ответ: t-test (pooled t-test, двухвыборочный t-тест Стьюдента)**

**Обоснование:** При неизвестных, но равных дисперсиях используется классический двухвыборочный t-тест с объединённой оценкой дисперсии (pooled variance). Статистика t = (X̄₁ - X̄₂) / (s_pooled * √(1/n₁ + 1/n₂)) имеет распределение Стьюдента с (n₁ + n₂ - 2) степенями свободы.

---

**3. Сравнить средние двух независимых групп при неизвестных и неравных дисперсиях.**

**Ответ: Welch t-test**

**Обоснование:** При неравных дисперсиях (гетероскедастичность) классический t-тест даёт искажённые результаты. Welch t-test использует скорректированные степени свободы по формуле Уэлча-Саттертуэйта и не требует предположения о равенстве дисперсий.

---

**4. Оценить, одинаковы ли средние в трёх и более группах.**

**Ответ: ANOVA (one-way)**

**Обоснование:** Однофакторный дисперсионный анализ (ANOVA) является обобщением t-теста для случая трёх и более групп. Проверяет гипотезу H₀: μ₁ = μ₂ = ... = μₖ против альтернативы, что хотя бы одно среднее отличается. Использует F-статистику, основанную на отношении межгрупповой и внутригрупповой дисперсий.

---

**5. Проверить, согласуются ли наблюдаемые частоты с распределением Пуассона.**

**Ответ: χ² (goodness-of-fit test, критерий согласия хи-квадрат)**

**Обоснование:** Критерий согласия хи-квадрат используется для проверки соответствия наблюдаемых частот теоретическому распределению (в данном случае — распределению Пуассона). Статистика χ² = Σ((Oᵢ - Eᵢ)² / Eᵢ) имеет распределение хи-квадрат с (k - 1 - r) степенями свободы, где k — число категорий, r — число оцениваемых параметров.

---

**Дополнительные статистики для уточнения:**

- **F-test (сравнение дисперсий):** Используется для проверки равенства дисперсий двух групп (тест Фишера) или нескольких групп (тест Бартлетта). Важен как предварительная проверка допущений для t-теста и ANOVA.

- **Levene:** Тест на равенство дисперсий, более устойчивый к отклонениям от нормальности, чем F-тест. Рекомендуется для проверки гомоскедастичности перед ANOVA.

- **Tukey HSD:** Пост-hoc тест для попарных сравнений средних после значимого результата ANOVA. Контролирует ошибку первого рода при множественных сравнениях.


## Задание 2. Тест для дозатора (z-test, двусторонний)

Инженерная задача. Дозатор рассчитывает среднюю дозу 3.00 г. Получено n=50 измерений со средним X̄=3.005 и известным стандартным отклонением процесса σ=0.015.

а) Проведите двусторонний z-тест для проверки H₀:μ=3.00 при α=0.05.

б) Постройте 95% доверительный интервал для μ.

в) Сделайте практический вывод для инженера: нужно ли перенастраивать дозатор?

**Требуется: формулы расчёта, численные значения (z_obs, p-value, CI), вывод.**


In [ ]:
mu0 = 3.00
xbar = 3.005
sigma = 0.015
n = 50
alpha = 0.05

print("=" * 80)
print("ЗАДАНИЕ 2: z-test для дозатора")
print("=" * 80)
print(f"Гипотеза H₀: μ = {mu0} г")
print(f"Гипотеза H₁: μ ≠ {mu0} г (двусторонний тест)")
print(f"Уровень значимости: α = {alpha}")
print(f"\nДанные:")
print(f"  Выборочное среднее: X̄ = {xbar} г")
print(f"  Известное стандартное отклонение: σ = {sigma} г")
print(f"  Объём выборки: n = {n}")

# а) Двусторонний z-test
print("\n" + "-" * 80)
print("а) Двусторонний z-test")
print("-" * 80)

# Стандартная ошибка среднего
se = sigma / np.sqrt(n)
print(f"Стандартная ошибка: SE = σ/√n = {sigma}/√{n} = {se:.6f} г")

# z-статистика
z_obs = (xbar - mu0) / se
print(f"\nz-статистика: z = (X̄ - μ₀) / SE = ({xbar} - {mu0}) / {se:.6f} = {z_obs:.4f}")

# p-value (двусторонний)
p_value = 2 * (1 - norm.cdf(abs(z_obs)))
print(f"p-value (двусторонний) = 2 * P(Z > |{z_obs:.4f}|) = {p_value:.6f}")

# Критическое значение
z_critical = norm.ppf(1 - alpha/2)
print(f"\nКритическое значение: z_{1-alpha/2} = {z_critical:.4f}")

# Вывод
print(f"\nВывод по z-test:")
if abs(z_obs) > z_critical:
    print(f"  |z_obs| = {abs(z_obs):.4f} > z_critical = {z_critical:.4f} => Отклоняем H₀")
else:
    print(f"  |z_obs| = {abs(z_obs):.4f} ≤ z_critical = {z_critical:.4f} => Не отклоняем H₀")

if p_value < alpha:
    print(f"  p-value = {p_value:.6f} < α = {alpha} => Отклоняем H₀")
else:
    print(f"  p-value = {p_value:.6f} ≥ α = {alpha} => Не отклоняем H₀")

# б) 95% доверительный интервал
print("\n" + "-" * 80)
print("б) 95% доверительный интервал для μ")
print("-" * 80)

confidence = 0.95
z_ci = norm.ppf(1 - (1 - confidence)/2)
ci_lower = xbar - z_ci * se
ci_upper = xbar + z_ci * se

print(f"Доверительный интервал: X̄ ± z_{1-(1-confidence)/2} * SE")
print(f"  = {xbar} ± {z_ci:.4f} * {se:.6f}")
print(f"  = [{ci_lower:.6f}, {ci_upper:.6f}] г")
print(f"\nС вероятностью {confidence*100}% истинное среднее значение μ находится в интервале [{ci_lower:.6f}, {ci_upper:.6f}] г")

# в) Практический вывод
print("\n" + "-" * 80)
print("в) Практический вывод для инженера")
print("-" * 80)

if p_value < alpha:
    print("РЕКОМЕНДАЦИЯ: Перенастроить дозатор.")
    print(f"Обоснование: Статистически значимое отклонение среднего значения ({xbar} г) от заданного ({mu0} г).")
    print(f"p-value = {p_value:.6f} < α = {alpha}, что указывает на систематическое отклонение.")
else:
    print("РЕКОМЕНДАЦИЯ: Перенастройка не требуется.")
    print(f"Обоснование: Статистически значимого отклонения не обнаружено (p-value = {p_value:.6f} ≥ α = {alpha}).")
    print(f"Наблюдаемое отклонение {abs(xbar - mu0):.4f} г находится в пределах случайной вариации.")
    print(f"Доверительный интервал [{ci_lower:.6f}, {ci_upper:.6f}] г включает заданное значение {mu0} г.")


## Задание 3. Сравнение прочности материалов (t-test vs Welch)

Даны две независимые выборки прочности материалов (в MPa):

**Группа A (n₁=15):** [51.99, 49.45, 52.59, 56.09, 49.06, 49.06, 56.32, 53.07, 48.12, 52.17, 48.15, 48.14, 50.97, 42.35, 43.10]

**Группа B (n₂=10):** [48.63, 45.92, 53.89, 46.55, 43.53, 60.79, 50.65, 52.41, 43.45, 48.73]

**Требуется:**

1. Проверить равенство дисперсий с помощью тестов Levene и классического F-test.
2. В зависимости от результата выполнить либо pooled t-test, либо Welch t-test для проверки H₀: μA=μB при α=0.05.
3. Построить 95% доверительный интервал для разности средних и интерпретировать.
4. Оформить заключение и вывод.


In [ ]:
A = np.array([51.99,49.45,52.59,56.09,49.06,49.06,56.32,53.07,48.12,52.17,48.15,48.14,50.97,42.35,43.10])
B = np.array([48.63,45.92,53.89,46.55,43.53,60.79,50.65,52.41,43.45,48.73])

print("=" * 80)
print("ЗАДАНИЕ 3: Сравнение прочности материалов")
print("=" * 80)
print(f"Группа A: n₁ = {len(A)}, среднее = {np.mean(A):.2f} MPa, std = {np.std(A, ddof=1):.2f} MPa")
print(f"Группа B: n₂ = {len(B)}, среднее = {np.mean(B):.2f} MPa, std = {np.std(B, ddof=1):.2f} MPa")

# Проверка равенства дисперсий
print("\n" + "-" * 80)
print("1. Проверка равенства дисперсий")
print("-" * 80)

# Levene test
levene_stat, levene_p = levene(A, B)
print(f"\nТест Левена:")
print(f"  Статистика: {levene_stat:.4f}")
print(f"  p-value: {levene_p:.6f}")
if levene_p < 0.05:
    print(f"  Вывод: Дисперсии различаются (p < 0.05)")
else:
    print(f"  Вывод: Дисперсии не различаются (p ≥ 0.05)")

# F-test (классический)
var_A = np.var(A, ddof=1)
var_B = np.var(B, ddof=1)
f_stat = var_A / var_B if var_A >= var_B else var_B / var_A
df1 = len(A) - 1
df2 = len(B) - 1
f_p = 2 * min(f.cdf(f_stat, df1, df2), 1 - f.cdf(f_stat, df1, df2))

print(f"\nКлассический F-test:")
print(f"  F-статистика: {f_stat:.4f}")
print(f"  p-value: {f_p:.6f}")
if f_p < 0.05:
    print(f"  Вывод: Дисперсии различаются (p < 0.05)")
else:
    print(f"  Вывод: Дисперсии не различаются (p ≥ 0.05)")

# Решение: какой тест использовать
equal_var = levene_p >= 0.05
print(f"\nРешение: {'Используем pooled t-test (равные дисперсии)' if equal_var else 'Используем Welch t-test (неравные дисперсии)'}")

# t-test
print("\n" + "-" * 80)
print("2. Сравнение средних")
print("-" * 80)

alpha = 0.05
print(f"Гипотеза H₀: μ_A = μ_B")
print(f"Гипотеза H₁: μ_A ≠ μ_B")
print(f"Уровень значимости: α = {alpha}")

# Pooled t-test
t_stat_pooled, p_pooled = ttest_ind(A, B, equal_var=True)
print(f"\nPooled t-test (равные дисперсии):")
print(f"  t-статистика: {t_stat_pooled:.4f}")
print(f"  p-value: {p_pooled:.6f}")

# Welch t-test
t_stat_welch, p_welch = ttest_ind(A, B, equal_var=False)
print(f"\nWelch t-test (неравные дисперсии):")
print(f"  t-статистика: {t_stat_welch:.4f}")
print(f"  p-value: {p_welch:.6f}")

# Выбор теста
if equal_var:
    t_final, p_final = t_stat_pooled, p_pooled
    test_name = "Pooled t-test"
else:
    t_final, p_final = t_stat_welch, p_welch
    test_name = "Welch t-test"

print(f"\nИспользуемый тест: {test_name}")
print(f"  t-статистика: {t_final:.4f}")
print(f"  p-value: {p_final:.6f}")

if p_final < alpha:
    print(f"  Вывод: Отклоняем H₀ (p < {alpha}) - средние различаются")
else:
    print(f"  Вывод: Не отклоняем H₀ (p ≥ {alpha}) - средние не различаются")

# Доверительный интервал для разности средних
print("\n" + "-" * 80)
print("3. 95% доверительный интервал для разности средних")
print("-" * 80)

mean_A = np.mean(A)
mean_B = np.mean(B)
diff = mean_A - mean_B

if equal_var:
    # Pooled standard error
    pooled_var = ((len(A)-1)*var_A + (len(B)-1)*var_B) / (len(A) + len(B) - 2)
    pooled_std = np.sqrt(pooled_var)
    se_diff = pooled_std * np.sqrt(1/len(A) + 1/len(B))
    df_ci = len(A) + len(B) - 2
else:
    # Welch standard error
    se_A = np.std(A, ddof=1) / np.sqrt(len(A))
    se_B = np.std(B, ddof=1) / np.sqrt(len(B))
    se_diff = np.sqrt(se_A**2 + se_B**2)
    # Welch-Satterthwaite degrees of freedom
    df_ci = (se_A**2 + se_B**2)**2 / (se_A**4/(len(A)-1) + se_B**4/(len(B)-1))

t_critical = t.ppf(0.975, df_ci)
ci_lower = diff - t_critical * se_diff
ci_upper = diff + t_critical * se_diff

print(f"Разность средних: X̄_A - X̄_B = {mean_A:.2f} - {mean_B:.2f} = {diff:.2f} MPa")
print(f"Стандартная ошибка разности: {se_diff:.4f} MPa")
print(f"Доверительный интервал: [{ci_lower:.4f}, {ci_upper:.4f}] MPa")

if ci_lower <= 0 <= ci_upper:
    print(f"\nИнтерпретация: Доверительный интервал включает 0, что согласуется с отсутствием значимого различия.")
else:
    print(f"\nИнтерпретация: Доверительный интервал не включает 0, что указывает на значимое различие средних.")

# Заключение
print("\n" + "=" * 80)
print("ЗАКЛЮЧЕНИЕ")
print("=" * 80)
print(f"1. Проверка дисперсий: {'Дисперсии равны' if equal_var else 'Дисперсии неравны'} (Levene p = {levene_p:.6f})")
print(f"2. Использован тест: {test_name}")
print(f"3. Результат: {'Средние различаются' if p_final < alpha else 'Средние не различаются'} (p = {p_final:.6f})")
print(f"4. Доверительный интервал для разности: [{ci_lower:.4f}, {ci_upper:.4f}] MPa")
print(f"\nПрактический вывод: {'Материалы имеют различную прочность' if p_final < alpha else 'Статистически значимого различия в прочности материалов не обнаружено'}.")


## Задание 4. Парные измерения — новая vs старая схема

Эксперимент парного дизайна: каждому прибору (или образцу) до и после изменения схемы измеряется показатель.

Даны пары (старое, новое):

(12.5,12.2),(13.1,13.0),(11.8,12.0),(12.9,12.7),(13.5,13.6),(12.0,11.9),(13.2,13.4)

а) Проверьте нормальность распределения разностей (Shapiro-Wilk).

б) Выполните парный t-test (или непараметрический Wilcoxon, если нормальность нарушена) для проверки, изменился ли показатель при новой схеме (α=0.05).

в) Постройте доверительный интервал для средней разности и вычислите парный Cohen's d.

**Требуется: расчёты, тест, p-value, CI, эффект и вывод.**


In [ ]:
old = np.array([12.5,13.1,11.8,12.9,13.5,12.0,13.2])
new = np.array([12.2,13.0,12.0,12.7,13.6,11.9,13.4])

print("=" * 80)
print("ЗАДАНИЕ 4: Парные измерения (старая vs новая схема)")
print("=" * 80)

differences = new - old
print(f"Разности (новое - старое): {differences}")
print(f"Средняя разность: {np.mean(differences):.4f}")
print(f"Стандартное отклонение разностей: {np.std(differences, ddof=1):.4f}")

# а) Проверка нормальности разностей
print("\n" + "-" * 80)
print("а) Проверка нормальности распределения разностей (Shapiro-Wilk)")
print("-" * 80)

shapiro_stat, shapiro_p = shapiro(differences)
print(f"Тест Шапиро-Уилка:")
print(f"  Статистика: {shapiro_stat:.4f}")
print(f"  p-value: {shapiro_p:.6f}")

is_normal = shapiro_p >= 0.05
if is_normal:
    print(f"  Вывод: Распределение разностей нормально (p ≥ 0.05)")
    print(f"  → Используем парный t-test")
else:
    print(f"  Вывод: Распределение разностей не нормально (p < 0.05)")
    print(f"  → Используем непараметрический тест Wilcoxon")

# Визуализация
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(differences, bins=5, edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Разность (новое - старое)')
axes[0].set_ylabel('Частота')
axes[0].set_title('Гистограмма разностей')
axes[0].axvline(0, color='red', linestyle='--', label='Нулевая разность')
axes[0].legend()

st.probplot(differences, dist="norm", plot=axes[1])
axes[1].set_title('Q-Q plot для проверки нормальности')

plt.tight_layout()
plt.show()

# б) Парный t-test или Wilcoxon
print("\n" + "-" * 80)
print("б) Тест для проверки изменения показателя")
print("-" * 80)

alpha = 0.05
print(f"Гипотеза H₀: μ_diff = 0 (показатель не изменился)")
print(f"Гипотеза H₁: μ_diff ≠ 0 (показатель изменился)")
print(f"Уровень значимости: α = {alpha}")

if is_normal:
    # Парный t-test
    t_stat, p_value = ttest_rel(new, old)
    print(f"\nПарный t-test:")
    print(f"  t-статистика: {t_stat:.4f}")
    print(f"  p-value: {p_value:.6f}")
    test_name = "Парный t-test"
else:
    # Wilcoxon signed-rank test
    wilcoxon_stat, p_value = wilcoxon(differences)
    print(f"\nТест Уилкоксона (Wilcoxon signed-rank):")
    print(f"  Статистика: {wilcoxon_stat:.4f}")
    print(f"  p-value: {p_value:.6f}")
    test_name = "Wilcoxon signed-rank test"

if p_value < alpha:
    print(f"  Вывод: Отклоняем H₀ (p < {alpha}) - показатель изменился")
else:
    print(f"  Вывод: Не отклоняем H₀ (p ≥ {alpha}) - показатель не изменился")

# в) Доверительный интервал и Cohen's d
print("\n" + "-" * 80)
print("в) Доверительный интервал для средней разности и Cohen's d")
print("-" * 80)

mean_diff = np.mean(differences)
std_diff = np.std(differences, ddof=1)
n = len(differences)

# Доверительный интервал
t_critical = t.ppf(0.975, n-1)
se_diff = std_diff / np.sqrt(n)
ci_lower = mean_diff - t_critical * se_diff
ci_upper = mean_diff + t_critical * se_diff

print(f"Средняя разность: {mean_diff:.4f}")
print(f"Стандартная ошибка: {se_diff:.4f}")
print(f"95% доверительный интервал: [{ci_lower:.4f}, {ci_upper:.4f}]")

# Cohen's d (парный)
cohens_d = mean_diff / std_diff
print(f"\nCohen's d (парный) = средняя разность / стандартное отклонение разностей")
print(f"  = {mean_diff:.4f} / {std_diff:.4f} = {cohens_d:.4f}")

# Интерпретация Cohen's d
if abs(cohens_d) < 0.2:
    effect_size = "незначительный"
elif abs(cohens_d) < 0.5:
    effect_size = "малый"
elif abs(cohens_d) < 0.8:
    effect_size = "средний"
else:
    effect_size = "большой"

print(f"  Интерпретация: {effect_size} эффект")

# Заключение
print("\n" + "=" * 80)
print("ЗАКЛЮЧЕНИЕ")
print("=" * 80)
print(f"1. Нормальность: {'Распределение нормально' if is_normal else 'Распределение не нормально'} (Shapiro p = {shapiro_p:.6f})")
print(f"2. Использован тест: {test_name}")
print(f"3. Результат: {'Показатель изменился' if p_value < alpha else 'Показатель не изменился'} (p = {p_value:.6f})")
print(f"4. Доверительный интервал: [{ci_lower:.4f}, {ci_upper:.4f}]")
print(f"5. Размер эффекта: Cohen's d = {cohens_d:.4f} ({effect_size} эффект)")
print(f"\nПрактический вывод: {'Новая схема статистически значимо изменяет показатель' if p_value < alpha else 'Статистически значимого изменения показателя не обнаружено'}.")


## Задание 5. До/после — медицинский пример (paired)

Клиническое исследование: измерен уровень глюкозы у 10 пациентов до и после приёма нового лекарства.

**Требуется:**

а) Проверить нормальность распределения разностей (тест Shapiro–Wilk).

б) Если нормальность не отвергается — выполнить парный t-test при α=0.01.

в) Если нормальность нарушена — выполнить непараметрический тест Wilcoxon signed-rank и сравнить выводы.

**Формат вывода: Код, p-values, выводы тестов, практическая интерпретация (снижается ли уровень глюкозы достоверно?).**


In [ ]:
before = np.array([7.1,6.8,7.5,6.9,7.3,6.5,7.0,6.7,7.2,6.9])
after  = np.array([6.6,6.4,7.1,6.5,7.2,6.3,6.6,6.4,7.0,6.4])

print("=" * 80)
print("ЗАДАНИЕ 5: Медицинский пример (уровень глюкозы до/после)")
print("=" * 80)

differences = after - before
print(f"Разности (после - до): {differences}")
print(f"Средняя разность: {np.mean(differences):.4f} ммоль/л")
print(f"Стандартное отклонение разностей: {np.std(differences, ddof=1):.4f} ммоль/л")

# а) Проверка нормальности
print("\n" + "-" * 80)
print("а) Проверка нормальности распределения разностей (Shapiro-Wilk)")
print("-" * 80)

shapiro_stat, shapiro_p = shapiro(differences)
print(f"Тест Шапиро-Уилка:")
print(f"  Статистика: {shapiro_stat:.4f}")
print(f"  p-value: {shapiro_p:.6f}")

is_normal = shapiro_p >= 0.01  # Уровень значимости 0.01
if is_normal:
    print(f"  Вывод: Распределение разностей нормально (p ≥ 0.01)")
    print(f"  → Используем парный t-test")
else:
    print(f"  Вывод: Распределение разностей не нормально (p < 0.01)")
    print(f"  → Используем непараметрический тест Wilcoxon")

# Визуализация
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot([1, 2], [before, after], 'o-', alpha=0.6, label='Индивидуальные значения')
axes[0].plot([1, 2], [np.mean(before), np.mean(after)], 's-', linewidth=2, markersize=10, label='Средние значения')
axes[0].set_xticks([1, 2])
axes[0].set_xticklabels(['До', 'После'])
axes[0].set_ylabel('Уровень глюкозы (ммоль/л)')
axes[0].set_title('Изменение уровня глюкозы')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

st.probplot(differences, dist="norm", plot=axes[1])
axes[1].set_title('Q-Q plot для проверки нормальности разностей')

plt.tight_layout()
plt.show()

# б) Парный t-test или Wilcoxon
print("\n" + "-" * 80)
print("б) Тест для проверки снижения уровня глюкозы")
print("-" * 80)

alpha = 0.01
print(f"Гипотеза H₀: μ_after = μ_before (уровень глюкозы не изменился)")
print(f"Гипотеза H₁: μ_after < μ_before (уровень глюкозы снизился)")
print(f"Уровень значимости: α = {alpha}")

if is_normal:
    # Парный t-test (односторонний: после < до)
    t_stat, p_value = ttest_rel(after, before, alternative='less')
    print(f"\nПарный t-test (односторонний):")
    print(f"  t-статистика: {t_stat:.4f}")
    print(f"  p-value: {p_value:.6f}")
    test_name = "Парный t-test (односторонний)"
else:
    # Wilcoxon signed-rank test (односторонний)
    wilcoxon_stat, p_value = wilcoxon(differences, alternative='less')
    print(f"\nТест Уилкоксона (Wilcoxon signed-rank, односторонний):")
    print(f"  Статистика: {wilcoxon_stat:.4f}")
    print(f"  p-value: {p_value:.6f}")
    test_name = "Wilcoxon signed-rank test (односторонний)"

if p_value < alpha:
    print(f"  Вывод: Отклоняем H₀ (p < {alpha}) - уровень глюкозы достоверно снизился")
else:
    print(f"  Вывод: Не отклоняем H₀ (p ≥ {alpha}) - достоверного снижения не обнаружено")

# в) Сравнение с двусторонним тестом
print("\n" + "-" * 80)
print("в) Сравнение с двусторонним тестом")
print("-" * 80)

if is_normal:
    t_stat_two, p_value_two = ttest_rel(after, before)
    print(f"Двусторонний парный t-test:")
    print(f"  p-value: {p_value_two:.6f}")
else:
    wilcoxon_stat_two, p_value_two = wilcoxon(differences)
    print(f"Двусторонний тест Уилкоксона:")
    print(f"  p-value: {p_value_two:.6f}")

# Заключение
print("\n" + "=" * 80)
print("ЗАКЛЮЧЕНИЕ")
print("=" * 80)
print(f"1. Нормальность: {'Распределение нормально' if is_normal else 'Распределение не нормально'} (Shapiro p = {shapiro_p:.6f})")
print(f"2. Использован тест: {test_name}")
print(f"3. Результат одностороннего теста: p = {p_value:.6f}")
print(f"4. Результат двустороннего теста: p = {p_value_two:.6f}")

if p_value < alpha:
    print(f"\nПрактическая интерпретация: Уровень глюкозы достоверно снижается после приёма лекарства (p = {p_value:.6f} < α = {alpha}).")
    print(f"Среднее снижение: {abs(np.mean(differences)):.4f} ммоль/л")
else:
    print(f"\nПрактическая интерпретация: Достоверного снижения уровня глюкозы не обнаружено (p = {p_value:.6f} ≥ α = {alpha}).")
    print(f"Наблюдаемое снижение {abs(np.mean(differences)):.4f} ммоль/л может быть случайным.")


## Задание 6. Welch и Games-Howell (симуляция с неравными дисперсиями)

Смоделируйте три группы с одинаковыми средними (например, 10), но существенно разными дисперсиями и неравными размерами выборок (пример: n1=20,n2=8,n3=5).

а) Покажите, что классическая ANOVA может ошибочно интерпретировать разницу, если нарушена гомоскедастичность.

б) Выполните Welch ANOVA и пост-hoc Games-Howell.

в) Сравните результаты и сделайте вывод о корректности подходов.

**Требуется: код симуляции, результаты ANOVA и Welch, таблица post-hoc и вывод.**


In [ ]:
print("=" * 80)
print("ЗАДАНИЕ 6: Welch ANOVA и Games-Howell (симуляция)")
print("=" * 80)

# Симуляция данных
np.random.seed(42)
n1, n2, n3 = 20, 8, 5
mean = 10  # Одинаковые средние
std1, std2, std3 = 1.0, 3.0, 5.0  # Разные дисперсии

group1 = np.random.normal(mean, std1, n1)
group2 = np.random.normal(mean, std2, n2)
group3 = np.random.normal(mean, std3, n3)

print(f"Симуляция данных:")
print(f"  Группа 1: n={n1}, μ={mean}, σ={std1} → среднее={np.mean(group1):.2f}, std={np.std(group1, ddof=1):.2f}")
print(f"  Группа 2: n={n2}, μ={mean}, σ={std2} → среднее={np.mean(group2):.2f}, std={np.std(group2, ddof=1):.2f}")
print(f"  Группа 3: n={n3}, μ={mean}, σ={std3} → среднее={np.mean(group3):.2f}, std={np.std(group3, ddof=1):.2f}")

# Визуализация
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

data_for_plot = [group1, group2, group3]
axes[0].boxplot(data_for_plot, labels=['Группа 1', 'Группа 2', 'Группа 3'])
axes[0].set_ylabel('Значение')
axes[0].set_title('Boxplots групп (разные дисперсии)')
axes[0].grid(True, alpha=0.3)

for i, group in enumerate(data_for_plot, 1):
    axes[1].scatter([i]*len(group), group, alpha=0.6, s=50)
    axes[1].axhline(np.mean(group), color=f'C{i-1}', linestyle='--', alpha=0.7, label=f'Среднее группы {i}')
axes[1].set_xticks([1, 2, 3])
axes[1].set_xticklabels(['Группа 1', 'Группа 2', 'Группа 3'])
axes[1].set_ylabel('Значение')
axes[1].set_title('Точечный график с средними')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# а) Классическая ANOVA
print("\n" + "-" * 80)
print("а) Классическая ANOVA (может ошибочно интерпретировать)")
print("-" * 80)

# Проверка гомоскедастичности
levene_stat, levene_p = levene(group1, group2, group3)
print(f"Тест Левена на равенство дисперсий:")
print(f"  Статистика: {levene_stat:.4f}")
print(f"  p-value: {levene_p:.6f}")
if levene_p < 0.05:
    print(f"  Вывод: Дисперсии различаются (p < 0.05) - допущение ANOVA нарушено!")
else:
    print(f"  Вывод: Дисперсии не различаются (p ≥ 0.05)")

# Классическая ANOVA
f_stat_classic, p_classic = st.f_oneway(group1, group2, group3)
print(f"\nКлассическая ANOVA:")
print(f"  F-статистика: {f_stat_classic:.4f}")
print(f"  p-value: {p_classic:.6f}")

# б) Welch ANOVA
print("\n" + "-" * 80)
print("б) Welch ANOVA (не требует равенства дисперсий)")
print("-" * 80)

if PINGOUIN_AVAILABLE:
    welch_result = pg.welch_anova(dv='value', between='group', data=pd.DataFrame({
        'value': np.concatenate([group1, group2, group3]),
        'group': ['G1']*n1 + ['G2']*n2 + ['G3']*n3
    }))
    f_welch = welch_result['F'].values[0]
    p_welch = welch_result['p-unc'].values[0]
    print(f"Welch ANOVA (pingouin):")
    print(f"  F-статистика: {f_welch:.4f}")
    print(f"  p-value: {p_welch:.6f}")
else:
    # Ручной расчет Welch ANOVA
    means = [np.mean(group1), np.mean(group2), np.mean(group3)]
    vars_ = [np.var(group1, ddof=1), np.var(group2, ddof=1), np.var(group3, ddof=1)]
    ns = [n1, n2, n3]
    
    # Общее среднее (взвешенное)
    grand_mean = np.sum([ns[i] * means[i] for i in range(3)]) / np.sum(ns)
    
    # Welch F-статистика
    numerator = np.sum([ns[i] * (means[i] - grand_mean)**2 for i in range(3)])
    denominator = np.sum([(1 - ns[i]/np.sum(ns)) * vars_[i] for i in range(3)])
    f_welch = numerator / denominator
    
    # Степени свободы (Welch-Satterthwaite)
    df_num = 2  # k-1
    df_denom = (np.sum([(1 - ns[i]/np.sum(ns))**2 * vars_[i]**2 / (ns[i] - 1) for i in range(3)]))**(-1)
    df_denom = 1 / df_denom
    
    p_welch = 1 - f.cdf(f_welch, df_num, df_denom)
    
    print(f"Welch ANOVA (ручной расчет):")
    print(f"  F-статистика: {f_welch:.4f}")
    print(f"  p-value: {p_welch:.6f}")
    print(f"  Степени свободы: ({df_num:.2f}, {df_denom:.2f})")

# в) Post-hoc Games-Howell
print("\n" + "-" * 80)
print("в) Post-hoc анализ: Games-Howell")
print("-" * 80)

if POSTHOCS_AVAILABLE:
    data_df = pd.DataFrame({
        'value': np.concatenate([group1, group2, group3]),
        'group': ['G1']*n1 + ['G2']*n2 + ['G3']*n3
    })
    
    posthoc = sp.posthoc_gameshowell(data_df, val_col='value', group_col='group')
    print("Таблица Games-Howell post-hoc:")
    print(posthoc)
else:
    print("scikit-posthocs не установлен. Используем альтернативный метод.")
    # Простое попарное сравнение с поправкой
    from itertools import combinations
    groups = [('G1', group1), ('G2', group2), ('G3', group3)]
    print("\nПопарные сравнения (Welch t-test с поправкой Бонферрони):")
    for (name1, data1), (name2, data2) in combinations(groups, 2):
        t_stat, p_val = ttest_ind(data1, data2, equal_var=False)
        p_corrected = min(p_val * 3, 1.0)  # Поправка Бонферрони для 3 сравнений
        print(f"  {name1} vs {name2}: t={t_stat:.4f}, p={p_val:.6f}, p_corrected={p_corrected:.6f}")

# Сравнение результатов
print("\n" + "=" * 80)
print("СРАВНЕНИЕ РЕЗУЛЬТАТОВ")
print("=" * 80)
print(f"1. Классическая ANOVA: F={f_stat_classic:.4f}, p={p_classic:.6f}")
print(f"2. Welch ANOVA: F={f_welch:.4f}, p={p_welch:.6f}")
print(f"\nВывод: Классическая ANOVA {'может давать искажённые результаты' if levene_p < 0.05 else 'применима'} при нарушении гомоскедастичности.")
print(f"Welch ANOVA корректно работает при неравных дисперсиях и даёт более точные результаты.")


## Задание 7. ANOVA на реальных данных (4 группы)

Данные эксперимента по влиянию четырёх типов корма на массу животных (в граммах). Каждая группа содержит по 6 наблюдений.

**Требуется:**

а) Выполнить графическую визуализацию (boxplots и точки). Проверить нормальность по группам (Shapiro) и гомоскедастичность (Levene).

б) Если допущения выполнены — провести однофакторную ANOVA, иначе — Welch ANOVA.

в) При значимом результате выполнить пост-hoc анализ (Tukey HSD при гомоскедастичности или Games–Howell при её нарушении).

г) Оценить размер эффекта η² и сделать практическую интерпретацию (какая кормовая формула наиболее эффективна).


In [ ]:
A = [251,262,248,255,260,257]
B = [263,270,265,272,260,266]
C = [268,275,271,280,277,269]
D = [282,290,285,295,288,293]

print("=" * 80)
print("ЗАДАНИЕ 7: ANOVA на реальных данных (4 группы корма)")
print("=" * 80)

groups = {'A': A, 'B': B, 'C': C, 'D': D}
for name, data in groups.items():
    print(f"Группа {name}: среднее={np.mean(data):.2f} г, std={np.std(data, ddof=1):.2f} г")

# а) Визуализация и проверка допущений
print("\n" + "-" * 80)
print("а) Визуализация и проверка допущений")
print("-" * 80)

# Визуализация
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

data_for_box = [A, B, C, D]
axes[0, 0].boxplot(data_for_box, labels=['A', 'B', 'C', 'D'])
axes[0, 0].set_ylabel('Масса (г)')
axes[0, 0].set_title('Boxplots по группам')
axes[0, 0].grid(True, alpha=0.3)

for i, (name, data) in enumerate(groups.items()):
    axes[0, 1].scatter([i]*len(data), data, alpha=0.6, s=80, label=f'Группа {name}')
    axes[0, 1].axhline(np.mean(data), color=f'C{i}', linestyle='--', alpha=0.7)
axes[0, 1].set_xticks([0, 1, 2, 3])
axes[0, 1].set_xticklabels(['A', 'B', 'C', 'D'])
axes[0, 1].set_ylabel('Масса (г)')
axes[0, 1].set_title('Точечный график с средними')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Проверка нормальности по группам
shapiro_results = {}
for name, data in groups.items():
    stat, p = shapiro(data)
    shapiro_results[name] = (stat, p)
    axes[1, 0].text(0.1, 0.9 - 0.2*list(groups.keys()).index(name), 
                   f'Группа {name}: Shapiro p={p:.4f}', transform=axes[1, 0].transAxes)

axes[1, 0].set_xlim(0, 1)
axes[1, 0].set_ylim(0, 1)
axes[1, 0].axis('off')
axes[1, 0].set_title('Результаты теста Шапиро-Уилка')

# Q-Q plots
for i, (name, data) in enumerate(groups.items()):
    st.probplot(data, dist="norm", plot=axes[1, 1])
axes[1, 1].set_title('Q-Q plots для всех групп')

plt.tight_layout()
plt.show()

print("\nПроверка нормальности (Shapiro-Wilk):")
all_normal = True
for name, (stat, p) in shapiro_results.items():
    is_normal = p >= 0.05
    all_normal = all_normal and is_normal
    print(f"  Группа {name}: статистика={stat:.4f}, p={p:.6f} {'(нормально)' if is_normal else '(не нормально)'}")

# Проверка гомоскедастичности
levene_stat, levene_p = levene(A, B, C, D)
print(f"\nПроверка гомоскедастичности (Levene):")
print(f"  Статистика: {levene_stat:.4f}")
print(f"  p-value: {levene_p:.6f}")
equal_var = levene_p >= 0.05
if equal_var:
    print(f"  Вывод: Дисперсии равны (p ≥ 0.05) - можно использовать классическую ANOVA")
else:
    print(f"  Вывод: Дисперсии неравны (p < 0.05) - используем Welch ANOVA")

# б) ANOVA
print("\n" + "-" * 80)
print("б) ANOVA")
print("-" * 80)

alpha = 0.05
print(f"Гипотеза H₀: μ_A = μ_B = μ_C = μ_D")
print(f"Гипотеза H₁: хотя бы одно среднее отличается")
print(f"Уровень значимости: α = {alpha}")

if equal_var and all_normal:
    # Классическая ANOVA
    f_stat, p_value = st.f_oneway(A, B, C, D)
    print(f"\nКлассическая ANOVA:")
    print(f"  F-статистика: {f_stat:.4f}")
    print(f"  p-value: {p_value:.6f}")
    anova_type = "Классическая ANOVA"
else:
    # Welch ANOVA
    if PINGOUIN_AVAILABLE:
        data_df = pd.DataFrame({
            'value': A + B + C + D,
            'group': ['A']*6 + ['B']*6 + ['C']*6 + ['D']*6
        })
        welch_result = pg.welch_anova(dv='value', between='group', data=data_df)
        f_stat = welch_result['F'].values[0]
        p_value = welch_result['p-unc'].values[0]
        print(f"\nWelch ANOVA:")
        print(f"  F-статистика: {f_stat:.4f}")
        print(f"  p-value: {p_value:.6f}")
    else:
        # Ручной расчет Welch
        all_groups = [A, B, C, D]
        means = [np.mean(g) for g in all_groups]
        vars_ = [np.var(g, ddof=1) for g in all_groups]
        ns = [len(g) for g in all_groups]
        
        grand_mean = np.sum([ns[i] * means[i] for i in range(4)]) / np.sum(ns)
        numerator = np.sum([ns[i] * (means[i] - grand_mean)**2 for i in range(4)])
        denominator = np.sum([(1 - ns[i]/np.sum(ns)) * vars_[i] for i in range(4)])
        f_stat = numerator / denominator
        
        df_num = 3
        df_denom = (np.sum([(1 - ns[i]/np.sum(ns))**2 * vars_[i]**2 / (ns[i] - 1) for i in range(4)]))**(-1)
        df_denom = 1 / df_denom
        
        p_value = 1 - f.cdf(f_stat, df_num, df_denom)
        print(f"\nWelch ANOVA (ручной расчет):")
        print(f"  F-статистика: {f_stat:.4f}")
        print(f"  p-value: {p_value:.6f}")
    anova_type = "Welch ANOVA"

if p_value < alpha:
    print(f"  Вывод: Отклоняем H₀ (p < {alpha}) - средние различаются")
else:
    print(f"  Вывод: Не отклоняем H₀ (p ≥ {alpha}) - средние не различаются")

# в) Post-hoc анализ
print("\n" + "-" * 80)
print("в) Post-hoc анализ")
print("-" * 80)

if p_value < alpha:
    if equal_var:
        # Tukey HSD
        data_for_tukey = pd.DataFrame({
            'value': A + B + C + D,
            'group': ['A']*6 + ['B']*6 + ['C']*6 + ['D']*6
        })
        tukey_result = pairwise_tukeyhsd(endog=data_for_tukey['value'], groups=data_for_tukey['group'], alpha=alpha)
        print("Tukey HSD post-hoc:")
        print(tukey_result)
        posthoc_type = "Tukey HSD"
    else:
        # Games-Howell
        if POSTHOCS_AVAILABLE:
            data_df = pd.DataFrame({
                'value': A + B + C + D,
                'group': ['A']*6 + ['B']*6 + ['C']*6 + ['D']*6
            })
            posthoc = sp.posthoc_gameshowell(data_df, val_col='value', group_col='group')
            print("Games-Howell post-hoc:")
            print(posthoc)
            posthoc_type = "Games-Howell"
        else:
            print("scikit-posthocs не установлен. Используем попарные Welch t-tests с поправкой Бонферрони.")
            from itertools import combinations
            groups_list = [('A', A), ('B', B), ('C', C), ('D', D)]
            n_comparisons = 6  # C(4,2) = 6
            print("\nПопарные сравнения (Welch t-test с поправкой Бонферрони):")
            for (name1, data1), (name2, data2) in combinations(groups_list, 2):
                t_stat, p_val = ttest_ind(data1, data2, equal_var=False)
                p_corrected = min(p_val * n_comparisons, 1.0)
                sig = "*" if p_corrected < alpha else ""
                print(f"  {name1} vs {name2}: t={t_stat:.4f}, p={p_val:.6f}, p_corrected={p_corrected:.6f} {sig}")
            posthoc_type = "Bonferroni-corrected Welch t-tests"
else:
    print("Post-hoc анализ не требуется (ANOVA не значима)")
    posthoc_type = "Не выполнен"

# г) Размер эффекта η²
print("\n" + "-" * 80)
print("г) Размер эффекта η²")
print("-" * 80)

# Расчет сумм квадратов
all_data = np.concatenate([A, B, C, D])
grand_mean = np.mean(all_data)

# Общая сумма квадратов (Q)
Q_total = np.sum((all_data - grand_mean)**2)

# Сумма квадратов между группами (Q₁)
means = [np.mean(A), np.mean(B), np.mean(C), np.mean(D)]
ns = [len(A), len(B), len(C), len(D)]
Q_between = np.sum([ns[i] * (means[i] - grand_mean)**2 for i in range(4)])

# Сумма квадратов внутри групп (Q₂)
Q_within = Q_total - Q_between

# η² (eta-squared)
eta_squared = Q_between / Q_total

print(f"Общая сумма квадратов (Q): {Q_total:.2f}")
print(f"Сумма квадратов между группами (Q₁): {Q_between:.2f}")
print(f"Сумма квадратов внутри групп (Q₂): {Q_within:.2f}")
print(f"\nη² = Q₁ / Q = {Q_between:.2f} / {Q_total:.2f} = {eta_squared:.4f}")

# Интерпретация η²
if eta_squared < 0.01:
    effect_interpretation = "очень малый"
elif eta_squared < 0.06:
    effect_interpretation = "малый"
elif eta_squared < 0.14:
    effect_interpretation = "средний"
else:
    effect_interpretation = "большой"

print(f"Интерпретация: {effect_interpretation} эффект ({eta_squared*100:.1f}% дисперсии объясняется фактором)")

# Заключение
print("\n" + "=" * 80)
print("ЗАКЛЮЧЕНИЕ")
print("=" * 80)
print(f"1. Допущения: нормальность={'выполнена' if all_normal else 'нарушена'}, гомоскедастичность={'выполнена' if equal_var else 'нарушена'}")
print(f"2. Использован тест: {anova_type}")
print(f"3. Результат ANOVA: F={f_stat:.4f}, p={p_value:.6f} - {'значимо' if p_value < alpha else 'не значимо'}")
print(f"4. Post-hoc: {posthoc_type}")
print(f"5. Размер эффекта: η²={eta_squared:.4f} ({effect_interpretation} эффект)")

# Практическая интерпретация
means_dict = {'A': np.mean(A), 'B': np.mean(B), 'C': np.mean(C), 'D': np.mean(D)}
best_group = max(means_dict, key=means_dict.get)
print(f"\nПрактическая интерпретация:")
print(f"Тип корма влияет на массу животных (η²={eta_squared:.4f}).")
print(f"Наиболее эффективная кормовая формула: группа {best_group} (средняя масса = {means_dict[best_group]:.2f} г).")


## Задание 8. Планирование эксперимента: размер выборки и мощность

Планируется эксперимент с l=4 группами. Требуется обеспечить мощность 1−β=0.8 для обнаружения разницы Δ=0.5σ между любыми парами групп (равный размер групп n), при α=0.05.

**Требуется:** 

а) Выписать формулу связи между размером эффекта (Cohen's f или η²), числом групп l и размером выборки n.

б) Приближённо оценить необходимый размер выборки для каждой группы при f≈0.25 (средний эффект по Cohen).

в) Привести практические рекомендации при ограниченных ресурсах (что можно изменить: число групп, эффект, α и т.п.).

**Подсказка: можно использовать модуль from statsmodels.stats.power import FTestAnovaPower**


In [ ]:
alpha = 0.05
power = 0.8
f = 0.25  # Cohen's f (средний эффект)
k = 4  # число групп

print("=" * 80)
print("ЗАДАНИЕ 8: Планирование эксперимента")
print("=" * 80)
print(f"Параметры:")
print(f"  Число групп: l = {k}")
print(f"  Уровень значимости: α = {alpha}")
print(f"  Мощность: 1-β = {power}")
print(f"  Размер эффекта: f = {f} (Cohen's f, средний эффект)")

# а) Формулы связи
print("\n" + "-" * 80)
print("а) Формулы связи между размером эффекта, числом групп и размером выборки")
print("-" * 80)

print("\nСвязь между f и η²:")
eta_squared = f**2 / (1 + f**2)
print(f"  f = √(η²/(1-η²))")
print(f"  η² = f²/(1+f²) = {f**2}/(1+{f**2}) = {eta_squared:.4f}")

print("\nФормула для размера выборки в ANOVA:")
print("  n = (2 * (z_α/2 + z_β)²) / (f²)  (приближенная для равных групп)")
print("  где:")
print("    z_α/2 - квантиль нормального распределения для уровня значимости")
print("    z_β - квантиль нормального распределения для мощности")
print("    f - размер эффекта Cohen's f")

# б) Расчет размера выборки
print("\n" + "-" * 80)
print("б) Расчет необходимого размера выборки")
print("-" * 80)

if hasattr(FTestAnovaPower, 'solve_power'):
    # Используем statsmodels
    power_analysis = FTestAnovaPower()
    n_per_group = power_analysis.solve_power(effect_size=f, alpha=alpha, power=power, k_groups=k)
    print(f"\nИспользуя statsmodels.stats.power.FTestAnovaPower:")
    print(f"  Необходимый размер выборки на группу: n = {n_per_group:.1f}")
    print(f"  Округление вверх: n = {int(np.ceil(n_per_group))}")
    print(f"  Общий размер выборки: N = {k} * {int(np.ceil(n_per_group))} = {k * int(np.ceil(n_per_group))}")
else:
    # Приближенный расчет
    z_alpha = norm.ppf(1 - alpha/2)
    z_beta = norm.ppf(power)
    n_approx = 2 * (z_alpha + z_beta)**2 / f**2
    print(f"\nПриближенный расчет:")
    print(f"  z_α/2 = {z_alpha:.4f}")
    print(f"  z_β = {z_beta:.4f}")
    print(f"  n ≈ 2 * ({z_alpha:.4f} + {z_beta:.4f})² / {f**2}")
    print(f"  n ≈ {n_approx:.1f}")
    print(f"  Округление вверх: n = {int(np.ceil(n_approx))}")
    n_per_group = int(np.ceil(n_approx))

# в) Практические рекомендации
print("\n" + "-" * 80)
print("в) Практические рекомендации при ограниченных ресурсах")
print("-" * 80)

print("\nПри ограниченных ресурсах можно изменить следующие параметры:")

print("\n1. Уровень значимости (α):")
print("   - Увеличение α (например, с 0.05 до 0.10) уменьшает необходимый размер выборки")
print("   - Но увеличивает вероятность ошибки первого рода (ложные положительные результаты)")
print("   - Рекомендация: использовать α=0.05 как стандарт, изменять только при обосновании")

print("\n2. Мощность (1-β):")
print("   - Уменьшение мощности (например, с 0.8 до 0.7) уменьшает размер выборки")
print("   - Но увеличивает вероятность ошибки второго рода (пропуск реального эффекта)")
print("   - Рекомендация: минимум 0.8, желательно 0.9 для важных исследований")

print("\n3. Размер эффекта (f):")
print("   - Если ожидается больший эффект, можно уменьшить размер выборки")
print("   - Но если эффект меньше ожидаемого, исследование может не обнаружить его")
print("   - Рекомендация: использовать консервативную оценку эффекта на основе пилотных исследований")

print("\n4. Число групп (l):")
print("   - Уменьшение числа групп уменьшает необходимый размер выборки")
print("   - Но ограничивает возможности сравнения")
print("   - Рекомендация: минимизировать число групп, оставив только необходимые для ответа на вопрос")

print("\n5. Неравные размеры групп:")
print("   - Можно использовать неравные размеры групп (например, больше наблюдений в контрольной группе)")
print("   - Это может быть более эффективно при ограниченных ресурсах")

print("\n6. Последовательный дизайн:")
print("   - Можно начать с меньшей выборки и увеличивать при необходимости")
print("   - Требует корректировки уровня значимости для множественных проверок")

# Примеры расчетов для разных параметров
print("\n" + "-" * 80)
print("Примеры расчетов для разных параметров:")
print("-" * 80)

scenarios = [
    ("Базовый", alpha, power, f, k),
    ("Увеличенный α", 0.10, power, f, k),
    ("Уменьшенная мощность", alpha, 0.7, f, k),
    ("Больший эффект", alpha, power, 0.4, k),
    ("Меньше групп", alpha, power, f, 3),
]

print(f"\n{'Сценарий':<20} {'α':<6} {'1-β':<6} {'f':<6} {'l':<4} {'n (на группу)':<15}")
print("-" * 70)

for name, a, p, eff, groups_count in scenarios:
    try:
        if hasattr(FTestAnovaPower, 'solve_power'):
            n_req = FTestAnovaPower().solve_power(effect_size=eff, alpha=a, power=p, k_groups=groups_count)
        else:
            z_a = norm.ppf(1 - a/2)
            z_p = norm.ppf(p)
            n_req = 2 * (z_a + z_p)**2 / eff**2
        print(f"{name:<20} {a:<6.2f} {p:<6.2f} {eff:<6.2f} {groups_count:<4} {int(np.ceil(n_req)):<15}")
    except:
        print(f"{name:<20} {a:<6.2f} {p:<6.2f} {eff:<6.2f} {groups_count:<4} {'N/A':<15}")

print("\n" + "=" * 80)
print("ЗАКЛЮЧЕНИЕ")
print("=" * 80)
print(f"Для обнаружения среднего эффекта (f={f}) при {k} группах с мощностью {power} и уровнем значимости {alpha}")
print(f"необходимо минимум {int(np.ceil(n_per_group))} наблюдений на группу (всего {k * int(np.ceil(n_per_group))} наблюдений).")
print(f"\nПри ограниченных ресурсах наиболее безопасным способом уменьшения размера выборки")
print(f"является уменьшение числа групп или увеличение ожидаемого размера эффекта (если это обосновано).")
